[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W10_Single_cells_and_biological_replication.ipynb)

# Week 10 | Single cells and biological replication

**Core practical: 45 minutes.** Aggregate counts by donor and cell type without erasing biological replication.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
Bulk RNA-seq averages mixtures. Single-cell measurements can separate cell populations, but cells remain nested within samples.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Who is submitting (1 min)
Fill in your **full name** and your **Ege student number**, then run the cell. It refuses to
continue if either is missing or malformed, so a mistyped digit is caught here — in the room,
where it takes ten seconds to fix — rather than after the deadline.

Your `COURSE_ID` above still deals your dataset. This is only about attributing the work to you.

In [ ]:
import os
if not os.path.exists("bib_colab.py"):
    !curl -sfO https://raw.githubusercontent.com/BMGLab/BFB/main/bib_colab.py
import bib_colab as bib

A = bib.start("W10")
A.whoami(
    name="",          # your full name, e.g. "Ayşe Gül Öztürk"
    student_no="",    # your Ege student number, digits only
    section="EN",     # "EN" or "TR"
)

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Inspect 32 synthetic cells from four donors and two cell types.
2. Sum counts within donor x cell-type groups.
3. Keep metadata for condition and donor.
4. Count independent donor replicates before and after aggregation.

In [ ]:
cells=[]
for donor in range(4):
    condition = "control" if donor < 2 else "treated"
    donor_effect = rng.randint(0,3)
    for celltype in ["T_cell","B_cell"]:
        for cell in range(4):
            cells.append({"donor":f"D{donor+1}","condition":condition,"celltype":celltype,
                          "G1":rng.randint(1,5)+donor_effect, "G2":rng.randint(0,5)})
pseudobulk={}
for c in cells:
    key=(c["donor"],c["celltype"])
    if key not in pseudobulk:
        pseudobulk[key]={"condition":c["condition"],"G1":0,"G2":0,"n_cells":0}
    for gene in ["G1","G2"]: pseudobulk[key][gene]+=c[gene]
    pseudobulk[key]["n_cells"]+=1
assert sum(c["G1"] for c in cells) == sum(p["G1"] for p in pseudobulk.values())
for key,value in pseudobulk.items(): print(key,value)
RESULTS={"data_status":"SYNTHETIC integer counts", "cells":len(cells),"pseudobulk_groups":len(pseudobulk),
         "donors_per_condition":2,"G1_total":sum(c["G1"] for c in cells)}
print(RESULTS)

## Explain the evidence (10 min)
**Q1.** Explain why 32 cells do not become 32 independent donors.

**Q2.** Which grouping keys preserve donor-level replication for a within-cell-type condition comparison?

**Q3.** Does successful aggregation establish adequate power or remove all batch confounding? Explain.

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, then **Runtime > Restart session and run all** so every number in
the notebook is the one your answers describe.

Then run the cell below. It checks that nothing is missing, prints a receipt code, and sends this
notebook straight to your instructor. There is nothing to download and nothing to upload.

A completion check looks for the presence of your responses, not for scientific correctness. If
the upload fails, the cell prints your receipt code and what to do instead — follow it before you
leave. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":10, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W10_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

# --- submit -------------------------------------------------------------
A.answers(RESPONSES,
          prediction=PREDICTION,
          check=CHECK_PERFORMED,
          disclosure=AI_DISCLOSURE,
          results=RESULTS)
A.check()
A.submit()

## Paper / device-free route
Four cells with G1 counts2,3,1,4 from one donor and cell type sum to10. Repeat for another donor; keep two separate columns, not one condition sum.

## Optional extension
Optional: differential abundance and paired donor designs; no atlas integration pipeline is required.

## Sources
- [S09] Love, Huber and Anders (2014). Moderated estimation of fold change and dispersion for RNA-seq data with DESeq2. https://doi.org/10.1186/s13059-014-0550-8
- [S12] Squair et al. (2021). Confronting false discoveries in single-cell differential expression. https://doi.org/10.1038/s41467-021-25960-2
- [S13] AnnData documentation: annotated data matrices. https://anndata.readthedocs.io/en/stable/
- [S27] EMBL-EBI: Single cell RNA-seq analysis using Python materials (2026). https://www.ebi.ac.uk/training/materials/single-cell-rna-seq-analysis-using-python-materials/